In [7]:
import vertexai
from google.adk.agents import Agent
from google.adk.tools import google_search
from vertexai import agent_engines
from vertexai.preview.reasoning_engines import AdkApp

# ==========================================
# 0. INITIALIZE VERTEX AI (Slide 63)
# ==========================================

PROJECT_ID = "qwiklabs-gcp-04-f81bee5d0509"
LOCATION = "us-central1"
STAGING_BUCKET = f"gs://{PROJECT_ID}-staging"

vertexai.init(
    project=PROJECT_ID,
    location=LOCATION,
    staging_bucket=STAGING_BUCKET
)

print(f"✅ Vertex AI initialized")
print(f"   Project: {PROJECT_ID}")
print(f"   Location: {LOCATION}")
print(f"   Staging bucket: {STAGING_BUCKET}\n")

# ==========================================
# 1. DEFINE THE AGENT (Slide 64)
# ==========================================
# NOTE: Using google_search (built-in ADK tool) matches the PDF example exactly.
# Built-in tools don't require external library imports, avoiding
# the cloudpickle serialization issues that occur with custom function tools.

search_agent = Agent(
    name="SearchAgent",
    model="gemini-2.5-flash",
    description="A helpful agent that searches the web for current information.",
    instruction="""
    You are a helpful assistant with the ability to search the web.
    Use the google_search tool to find accurate, up-to-date information
    to answer the user's questions. Summarize your findings clearly.
    """  ,
    tools=[google_search]
)

print("✅ Agent defined successfully\n")

# ==========================================
# 2. TEST LOCALLY FIRST (Slide 65)
# ==========================================

print("--- Step 1: Testing Agent Locally ---\n")

app = AdkApp(agent=search_agent)

for event in app.stream_query(
    user_id="local-test-user",
    message="Give me the news highlights in the world of sports."
):
    if "content" in event and "parts" in event["content"]:
        for part in event["content"]["parts"]:
            text = part.text if hasattr(part, 'text') else part.get("text", "")
            if text:
                print(f"Local Output: {text}")

print("\n✅ Local test complete!\n")

# ==========================================
# 3. DEPLOY TO AGENT ENGINE (Slide 66)
# ==========================================
# Exactly matches the PDF example:
# remote_agent = agent_engines.create(
#     app,
#     requirements=["google-cloud-aiplatform[agent_engines,adk]"],
# )

print("--- Step 2: Deploying Agent to Agent Engine ---")
print("⏳ This may take 3-5 minutes...\n")

remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[agent_engines,adk]"],
    display_name="Challenge5-SearchAgent",
    description="A search agent deployed to Agent Engine for the ADK Skills Workshop."
)

print(f"\n✅ Agent deployed successfully!")
print(f"Resource name: {remote_agent.resource_name}\n")

# ==========================================
# 4. TEST THE DEPLOYED AGENT (Slide 66)
# ==========================================

print("--- Step 3: Testing Deployed Agent on Agent Engine ---\n")

test_queries = [
    "Give me the news highlights in the world of sports.",
    "What are the latest developments in AI today?"
]

for message in test_queries:
    print(f"Query: '{message}'")

    for event in remote_agent.stream_query(
        user_id="agent-engine-test-user",
        message=message
    ):
        if "content" in event and "parts" in event["content"]:
            for part in event["content"]["parts"]:
                text = part.text if hasattr(part, 'text') else part.get("text", "")
                if text:
                    print(f"Cloud Agent Output: {text}")

    print("-" * 50)

# ==========================================
# 5. SAVE RESOURCE NAME FOR CLEANUP
# ==========================================

print(f"\n📝 Resource name (save for cleanup):")
print(f"{remote_agent.resource_name}")

# To delete when done (avoids ongoing charges):
# remote_agent.delete()

✅ Vertex AI initialized
   Project: qwiklabs-gcp-04-f81bee5d0509
   Location: us-central1
   Staging bucket: gs://qwiklabs-gcp-04-f81bee5d0509-staging

✅ Agent defined successfully

--- Step 1: Testing Agent Locally ---



INFO:vertexai.agent_engines:Identified the following requirements: {'cloudpickle': '3.1.2', 'google-cloud-aiplatform': '1.135.0', 'pydantic': '2.12.3'}
INFO:vertexai.agent_engines:The following requirements are appended: {'pydantic==2.12.3', 'cloudpickle==3.1.2'}
INFO:vertexai.agent_engines:The final list of requirements: ['google-cloud-aiplatform[agent_engines,adk]', 'pydantic==2.12.3', 'cloudpickle==3.1.2']
INFO:vertexai.agent_engines:Using bucket qwiklabs-gcp-04-f81bee5d0509-staging


Local Output: Here's a roundup of recent sports highlights:

**Football (Soccer):**

In the Premier League, there's a tight race at the top, with Arsenal extending their lead after defeating Chelsea. Manchester United also climbed to third place with a comeback win against Crystal Palace. Middlesbrough secured a vital win in their promotion race with a brace from Targett. Elsewhere, Lionel Messi scored twice to help Inter Miami come from behind to beat Orlando. Sevilla also managed a derby draw, impacting Real Betis's top-four aspirations.

**Cricket:**

The T20 World Cup has seen significant action. Sanju Samson has been a standout performer, with his unbeaten 97 leading India into the T20 World Cup semi-finals and breaking Virat Kohli's record for the highest score by an Indian batter while chasing in the tournament. India also registered a historic T20 World Cup win against West Indies. In other news, Jasprit Bumrah has been highlighted for his impactful performance, and Rohit Sharm

INFO:vertexai.agent_engines:Wrote to gs://qwiklabs-gcp-04-f81bee5d0509-staging/agent_engine/agent_engine.pkl
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-f81bee5d0509-staging/agent_engine/requirements.txt
INFO:vertexai.agent_engines:Creating in-memory tarfile of extra_packages
INFO:vertexai.agent_engines:Writing to gs://qwiklabs-gcp-04-f81bee5d0509-staging/agent_engine/dependencies.tar.gz
INFO:vertexai.agent_engines:Creating AgentEngine
INFO:vertexai.agent_engines:Create AgentEngine backing LRO: projects/163919411957/locations/us-central1/reasoningEngines/6427770264745934848/operations/7433376598713171968
INFO:vertexai.agent_engines:View progress and logs at https://console.cloud.google.com/logs/query?project=qwiklabs-gcp-04-f81bee5d0509
INFO:vertexai.agent_engines:AgentEngine created. Resource name: projects/163919411957/locations/us-central1/reasoningEngines/6427770264745934848
INFO:vertexai.agent_engines:To use this AgentEngine in another session:
INFO:vertexai.agent_


✅ Agent deployed successfully!
Resource name: projects/163919411957/locations/us-central1/reasoningEngines/6427770264745934848

--- Step 3: Testing Deployed Agent on Agent Engine ---

Query: 'Give me the news highlights in the world of sports.'
Cloud Agent Output: Here's a roundup of recent sports highlights from around the world:

**Football (Soccer)**
*   Lamine Yamal scored his first career hat-trick for Barcelona in their victory against Villarreal.
*   Sixteen-year-old Cavan Sullivan became the youngest American to score in the CONCACAF Champions Cup.
*   USA striker Josh Sargent transferred from Norwich to Toronto, concluding a "messy transfer saga".
*   Neymar boosted his case for the World Cup squad with two milestone goals for Santos.
*   Lionel Messi netted a double as Inter Miami battled back to defeat Orlando City 4-2.
*   Sevilla secured a derby draw, impacting Real Betis's top-four aspirations.
*   Arsenal claimed a nervy win over Chelsea, while Manchester United climbed

In [ ]:
# Re-try deployment with corrected requirements
# We use 'reasoningengine' extra which is the standard for Agent Engine

print("--- Retrying Deployment with updated requirements ---")
print("⏳ This may take 3-5 minutes...\n")

remote_agent = agent_engines.create(
    app,
    requirements=["google-cloud-aiplatform[reasoningengine]", "requests"],
    display_name="Challenge5-SearchAgent",
    description="A search agent deployed to Agent Engine for the ADK Skills Workshop."
)

print(f"\n✅ Agent deployed successfully!")
print(f"Resource name: {remote_agent.resource_name}\n")